# Cupo · el Airbnb de los estacionamientos
**Entrega 2 · Implementación del sistema** — Matías Pacheco, Francisco Auladell y Diego Balcázar

Este cuaderno ejecuta el sistema en Python con menú en consola. Todos los datos (estacionamientos, anfitriones, conductores, reservas, historial y calificaciones) se guardan en **`cupo_datos.xlsx`**, dentro de la carpeta **Mi unidad › Cupo** de Google Drive. Así no se pierden al cerrar Colab.

Si conectas una cuenta de Google, cada reserva crea un evento real en **Google Calendar** e invita por correo al conductor, al anfitrión y a quien agregues.

**Cómo usarlo:** ejecuta las celdas en orden con ▶. El paso 4 se hace una sola vez y es opcional.

## Paso 1 · Conectar Google Drive
Aquí se guardan el Excel, el código y el permiso de Google Calendar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
CARPETA = '/content/drive/MyDrive/Cupo'
os.makedirs(CARPETA, exist_ok=True)
if CARPETA not in sys.path:
    sys.path.insert(0, CARPETA)
print('Carpeta del proyecto:', CARPETA)

## Paso 2 · Instalar las librerías

In [ ]:
!pip install -q google-api-python-client google-auth-oauthlib openpyxl pandas

## Paso 3 · Guardar el código del sistema
Cada celda crea un módulo en la carpeta del proyecto: `modelos.py` (clases), `reglas.py` (reglas de negocio y las seis funcionalidades), `datos.py` (Excel y CSV), `calendario.py` (Google Calendar), `datos_demo.py` (datos de prueba) y `main.py` (menú).

In [ ]:
%%writefile /content/drive/MyDrive/Cupo/modelos.py
"""
modelos.py — Clases del sistema Cupo (el Airbnb de los estacionamientos).

Cada clase representa una entidad del negocio. Los datos de todas ellas
se guardan en una hoja del Excel (ver datos.py).
"""
from dataclasses import dataclass
from datetime import datetime

FORMATO_FECHA = "%Y-%m-%d %H:%M"
TAMANOS = {"pequeño": 1, "mediano": 2, "grande": 3}
DIAS_SEMANA = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]   # índice = datetime.weekday()


def a_fecha(texto):
    """Convierte 'AAAA-MM-DD HH:MM' en datetime."""
    return datetime.strptime(texto, FORMATO_FECHA)


def a_texto(fecha):
    """Convierte un datetime en 'AAAA-MM-DD HH:MM'."""
    return fecha.strftime(FORMATO_FECHA)


# ---------------------------------------------------------------- Usuarios
@dataclass
class Usuario:
    """Clase base de las tres personas que usan el sistema."""
    id: str
    nombre: str
    email: str = ""

    def rol(self):
        return "usuario"

    def __str__(self):
        return f"{self.nombre} ({self.rol()})"


@dataclass
class Anfitrion(Usuario):
    """Propietario: presta su estacionamiento cuando no lo usa."""
    departamento: str = ""

    def rol(self):
        return "anfitrión"


@dataclass
class Conductor(Usuario):
    """Cliente: reserva estacionamientos por horas."""
    patente: str = ""
    tamano_auto: str = "mediano"
    fechas_no_llegada: str = ""      # fechas separadas por ';' (para la regla de 30 días)
    en_revision: bool = False

    def rol(self):
        return "conductor"

    def lista_no_llegadas(self):
        return [x for x in self.fechas_no_llegada.split(";") if x]

    def agregar_no_llegada(self, fecha_texto):
        self.fechas_no_llegada = ";".join(self.lista_no_llegadas() + [fecha_texto])


@dataclass
class Administrador(Usuario):
    """Administración del edificio: aprueba, asigna y aplica penalizaciones."""

    def rol(self):
        return "administración"


# ---------------------------------------------------------------- Espacios
@dataclass
class Estacionamiento:
    id: str
    anfitrion_id: str
    edificio: str
    direccion: str
    nivel: str
    tamano: str = "mediano"          # tamaño máximo de auto que cabe
    techado: bool = True
    cargador: bool = False
    estado: str = "liberado"         # liberado | bloqueado
    motivo_bloqueo: str = ""
    aprobado: bool = True
    en_revision: bool = False
    hora_desde: str = "09:00"        # calendario inteligente (horario recurrente)
    hora_hasta: str = "18:00"
    dias: str = "0,1,2,3,4"          # 0 = lunes … 6 = domingo
    reglas: str = ""

    def lista_dias(self):
        return [int(d) for d in self.dias.split(",") if d.strip() != ""]

    def admite(self, tamano_auto):
        return TAMANOS[tamano_auto] <= TAMANOS[self.tamano]

    def ofrece(self, inicio, termino):
        """¿El horario recurrente del anfitrión cubre ese período?"""
        if inicio.date() != termino.date():
            return False
        if inicio.weekday() not in self.lista_dias():
            return False
        return self.hora_desde <= inicio.strftime("%H:%M") and termino.strftime("%H:%M") <= self.hora_hasta

    def describir(self):
        extras = []
        if self.techado:
            extras.append("techado")
        if self.cargador:
            extras.append("cargador")
        dias = ", ".join(DIAS_SEMANA[d] for d in self.lista_dias())
        return (f"{self.id} · {self.edificio}, nivel {self.nivel} · auto {self.tamano}"
                f"{' · ' + ', '.join(extras) if extras else ''} · {dias} {self.hora_desde}-{self.hora_hasta}")


# ---------------------------------------------------------------- Reservas
@dataclass
class Reserva:
    id: str
    conductor_id: str
    conductor_nombre: str
    inicio: str                      # 'AAAA-MM-DD HH:MM'
    termino: str
    estacionamiento_id: str = ""     # vacío mientras espera asignación
    anfitrion_id: str = ""
    estado: str = "confirmada"       # por_asignar | confirmada | en_uso | finalizada | cancelada | no_llego
    monto: int = 0
    cobro: int = 0
    devolucion: int = 0
    recargo: int = 0
    codigo_acceso: str = ""
    creada: str = ""
    llegada: str = ""
    fin_uso: str = ""
    calificacion: int = 0
    invitados: str = ""              # correos separados por ','
    evento_google: str = ""          # id del evento en Google Calendar
    motivo: str = ""

    ACTIVAS = ("confirmada", "en_uso")

    def activa(self):
        return self.estado in self.ACTIVAS

    def se_superpone(self, inicio, termino):
        return a_fecha(self.inicio) < termino and inicio < a_fecha(self.termino)

    def minutos(self):
        return int((a_fecha(self.termino) - a_fecha(self.inicio)).total_seconds() // 60)


# ---------------------------------------------------------------- Registro
@dataclass
class Evento:
    """Una fila del registro histórico (base del dashboard de la Entrega 3)."""
    fecha_hora: str
    tipo: str                        # creación, cancelación, liberación, reasignación, …
    reserva_id: str = ""
    estacionamiento_id: str = ""
    anfitrion_id: str = ""
    conductor_id: str = ""
    estado_reserva: str = ""
    inicio: str = ""
    termino: str = ""
    monto: int = 0
    detalle: str = ""


@dataclass
class Calificacion:
    fecha_hora: str
    reserva_id: str
    estacionamiento_id: str
    conductor_id: str
    estrellas: int
    comentario: str = ""

In [ ]:
%%writefile /content/drive/MyDrive/Cupo/reglas.py
"""
reglas.py — Lógica del negocio: las seis funcionalidades y sus validaciones.

La clase Sistema es el único punto que cambia datos. Cada operación:
  1) valida las reglas de negocio (RN01–RN05),
  2) actualiza estados de espacios y reservas,
  3) deja una fila en el registro histórico,
  4) guarda todo en el Excel.
"""
import random
from datetime import timedelta

from modelos import (Anfitrion, Conductor, Calificacion, Estacionamiento, Evento,
                     Reserva, a_fecha, a_texto)


class ErrorNegocio(Exception):
    """Operación inválida: el mensaje explica qué regla se rompió."""


class Sistema:
    # Parámetros del modelo de negocio (Entrega 1)
    PRECIO_BASE = 700            # por bloque de 30 min
    PRECIO_PEAK = 900            # bloques que empiezan entre 15:00 y 17:00
    TOPE_DIARIO = 8000
    RECARGO_MINUTO = 30
    TOLERANCIA_MIN = 15
    MAXIMO_MIN = 120             # reservas de hasta 2 h (riesgo 0,53 %)
    COBRO_NO_LLEGADA = 0.30
    NO_LLEGADAS_REVISION = 3     # en 30 días
    REPARTO = {"anfitrión": 0.60, "edificio": 0.25, "plataforma": 0.15}
    HORARIO_SUGERIDO = ("09:00", "18:00")   # sale del modelo D(t)

    def __init__(self, repositorio, calendario=None):
        self.repo = repositorio
        self.calendario = calendario
        datos = repositorio.cargar()
        self.anfitriones = {a.id: a for a in datos["Anfitriones"]}
        self.conductores = {c.id: c for c in datos["Conductores"]}
        self.administradores = {a.id: a for a in datos["Administradores"]}
        self.espacios = {e.id: e for e in datos["Estacionamientos"]}
        self.reservas = {r.id: r for r in datos["Reservas"]}
        self.historial = datos["Historial"]
        self.calificaciones = datos["Calificaciones"]
        self.config = datos["Config"]
        self.avisos = []        # mensajes de Google Calendar para mostrar en pantalla

    # ------------------------------------------------------------ utilidades
    def ahora(self):
        return a_fecha(self.config["reloj"])

    def guardar(self):
        self.repo.guardar(self)

    def _nuevo_id(self, clave, prefijo):
        n = int(self.config.get(clave, 1))
        self.config[clave] = str(n + 1)
        return f"{prefijo}{n:04d}"

    def _registrar(self, tipo, reserva=None, espacio_id="", detalle="", monto=0):
        r = reserva
        self.historial.append(Evento(
            fecha_hora=self.config["reloj"], tipo=tipo,
            reserva_id=r.id if r else "",
            estacionamiento_id=(r.estacionamiento_id if r else espacio_id) or espacio_id,
            anfitrion_id=(r.anfitrion_id if r else self.espacios[espacio_id].anfitrion_id if espacio_id in self.espacios else ""),
            conductor_id=r.conductor_id if r else "",
            estado_reserva=r.estado if r else "",
            inicio=r.inicio if r else "", termino=r.termino if r else "",
            monto=int(monto), detalle=detalle))

    def precio(self, inicio, termino):
        """Suma bloques de 30 min (peak de 15:00 a 17:00) con tope diario."""
        total, t = 0, inicio
        while t < termino:
            total += self.PRECIO_PEAK if 15 <= t.hour < 17 else self.PRECIO_BASE
            t += timedelta(minutes=30)
        return min(total, self.TOPE_DIARIO)

    def motivo_no_disponible(self, espacio, inicio, termino, tamano_auto, ignorar_reserva=""):
        """RN02 y RN03: devuelve por qué no se puede usar, o None si está disponible."""
        if not espacio.aprobado:
            return "espera aprobación de la administración"
        if espacio.en_revision:
            return "está en revisión por calificaciones bajas"
        if espacio.estado == "bloqueado":
            return "el dueño lo bloqueó para usarlo (prioridad del propietario)"
        if not espacio.ofrece(inicio, termino):
            return f"el anfitrión lo ofrece {espacio.hora_desde}-{espacio.hora_hasta} en sus días definidos"
        if not espacio.admite(tamano_auto):
            return f"es para autos hasta {espacio.tamano}"
        for r in self.reservas.values():
            if (r.estacionamiento_id == espacio.id and r.id != ignorar_reserva
                    and r.activa() and r.se_superpone(inicio, termino)):
                return f"ya está reservado de {r.inicio[11:]} a {r.termino[11:]}"
        return None

    def espacios_compatibles(self, inicio, termino, tamano_auto, ignorar_reserva="", excluir=""):
        return [e for e in self.espacios.values()
                if e.id != excluir and not self.motivo_no_disponible(e, inicio, termino, tamano_auto, ignorar_reserva)]

    def _validar_periodo(self, inicio, minutos):
        if minutos <= 0 or minutos % 30 != 0:
            raise ErrorNegocio("La duración debe ser en bloques de 30 minutos.")
        if minutos > self.MAXIMO_MIN:
            raise ErrorNegocio("El máximo por reserva es de 2 horas: sobre eso el riesgo de que vuelva el dueño sube.")
        if inicio < self.ahora():
            raise ErrorNegocio("No se puede reservar en una hora que ya pasó.")
        return inicio + timedelta(minutes=minutos)

    # ------------------------------------------------------------ Google Calendar
    def _cal_crear(self, r):
        if not r.estacionamiento_id:
            return
        e = self.espacios[r.estacionamiento_id]
        c = self.conductores[r.conductor_id]
        a = self.anfitriones[e.anfitrion_id]
        if self.calendario and self.calendario.conectado:
            try:
                r.evento_google = self.calendario.crear_evento(r, e, a, c)
                self.avisos.append(f"Evento creado en Google Calendar e invitaciones enviadas a: {', '.join(self.calendario.invitados(r, a, c))}")
                return
            except Exception as err:  # Calendar no debe impedir la reserva
                self.avisos.append(f"No se pudo crear el evento en Google Calendar ({err}).")
        if self.calendario:
            self.avisos.append("Enlace para agregarla a Google Calendar:\n" + self.calendario.enlace_plantilla(r, e, a, c))

    def _cal_actualizar(self, r):
        if r.evento_google and self.calendario and self.calendario.conectado:
            try:
                e = self.espacios[r.estacionamiento_id]
                self.calendario.actualizar_evento(r, e, self.anfitriones[e.anfitrion_id], self.conductores[r.conductor_id])
                self.avisos.append("Evento de Google Calendar actualizado: los invitados reciben el cambio por correo.")
                return
            except Exception as err:
                self.avisos.append(f"No se pudo actualizar el evento de Google Calendar ({err}).")
        elif not r.evento_google:
            self._cal_crear(r)

    def _cal_cancelar(self, r):
        if r.evento_google and self.calendario and self.calendario.conectado:
            try:
                self.calendario.cancelar_evento(r.evento_google)
                self.avisos.append("Evento eliminado de Google Calendar: los invitados reciben la cancelación.")
            except Exception as err:
                self.avisos.append(f"No se pudo eliminar el evento de Google Calendar ({err}).")

    # ============================================================ PROPIETARIO
    def _espacio_de(self, espacio_id, anfitrion_id):
        e = self.espacios.get(espacio_id)
        if not e:
            raise ErrorNegocio(f"No existe el estacionamiento {espacio_id}.")
        if e.anfitrion_id != anfitrion_id:
            raise ErrorNegocio(f"{espacio_id} no pertenece a este anfitrión.")
        return e

    def liberar_espacio(self, espacio_id, anfitrion_id):
        """Obligatoria propietario (1/2): ofrecer el espacio a terceros."""
        e = self._espacio_de(espacio_id, anfitrion_id)
        if e.estado == "liberado":
            raise ErrorNegocio(f"{espacio_id} ya está liberado.")
        e.estado, e.motivo_bloqueo = "liberado", ""
        self._registrar("liberación del espacio", espacio_id=e.id, detalle="El anfitrión vuelve a ofrecer su espacio")
        self.guardar()

    def bloquear_espacio(self, espacio_id, anfitrion_id, motivo="Uso del dueño"):
        """Obligatoria propietario (2/2). RN01: el dueño tiene prioridad.
        Contingencia: si hay reservas vigentes, se reubican en un espacio compatible;
        si no hay ninguno, se cancelan con devolución del 100 %."""
        e = self._espacio_de(espacio_id, anfitrion_id)
        if e.estado == "bloqueado":
            raise ErrorNegocio(f"{espacio_id} ya está bloqueado.")
        e.estado, e.motivo_bloqueo = "bloqueado", motivo
        self._registrar("bloqueo", espacio_id=e.id, detalle=motivo)
        resultado = []
        afectadas = [r for r in self.reservas.values()
                     if r.estacionamiento_id == e.id and r.activa() and a_fecha(r.termino) > self.ahora()]
        for r in afectadas:
            tamano = self.conductores[r.conductor_id].tamano_auto
            opciones = self.espacios_compatibles(a_fecha(r.inicio), a_fecha(r.termino), tamano, r.id, e.id)
            if opciones:
                nuevo = opciones[0]
                r.estacionamiento_id, r.anfitrion_id = nuevo.id, nuevo.anfitrion_id
                r.motivo = f"Reubicada desde {e.id}: el dueño recuperó su espacio"
                self._registrar("reasignación", r, detalle=r.motivo)
                self._cal_actualizar(r)
                resultado.append(f"{r.id} de {r.conductor_nombre}: reubicada en {nuevo.id}")
            else:
                r.estado, r.devolucion, r.motivo = "cancelada", r.monto, "El dueño recuperó su espacio y no había otro compatible"
                self._registrar("cancelación", r, detalle=r.motivo + ": devolución del 100 %")
                self._cal_cancelar(r)
                resultado.append(f"{r.id} de {r.conductor_nombre}: cancelada con devolución del 100 %")
        self.guardar()
        return resultado

    def definir_horario(self, espacio_id, anfitrion_id, desde, hasta, dias):
        """Propia propietario: calendario inteligente (horario recurrente).
        Valida el horario y advierte si sale de la ventana segura del modelo D(t)."""
        e = self._espacio_de(espacio_id, anfitrion_id)
        if desde >= hasta:
            raise ErrorNegocio("La hora de inicio debe ser anterior a la de término.")
        if not dias:
            raise ErrorNegocio("Elige al menos un día.")
        vigentes = [r for r in self.reservas.values() if r.estacionamiento_id == e.id and r.activa()]
        for r in vigentes:
            ini, fin = a_fecha(r.inicio), a_fecha(r.termino)
            if ini.weekday() not in dias or r.inicio[11:] < desde or r.termino[11:] > hasta:
                raise ErrorNegocio(f"El nuevo horario deja fuera la reserva {r.id} ({r.inicio} a {r.termino[11:]}). "
                                   "Para eso usa 'Bloquear', que reubica al conductor.")
        e.hora_desde, e.hora_hasta, e.dias = desde, hasta, ",".join(str(d) for d in sorted(dias))
        self._registrar("horario", espacio_id=e.id, detalle=f"{e.dias} de {desde} a {hasta}")
        self.guardar()
        avisos = []
        if desde < "08:30":
            avisos.append("Antes de las 08:30 la mayoría de los dueños aún no sale (D'(t) > 0): más riesgo de conflicto.")
        if hasta > "18:00":
            avisos.append("Después de las 18:00 los dueños vuelven (D'(t) < 0): más riesgo de conflicto.")
        return avisos

    def ganancia_estimada(self, espacio_id, ocupacion=0.60):
        """Simulador: bloques ofrecidos al mes × precio × ocupación × 60 % del anfitrión."""
        e = self.espacios[espacio_id]
        t = a_fecha("2000-01-03 " + e.hora_desde)
        fin = a_fecha("2000-01-03 " + e.hora_hasta)
        por_dia = 0
        while t < fin:
            por_dia += self.PRECIO_PEAK if 15 <= t.hour < 17 else self.PRECIO_BASE
            t += timedelta(minutes=30)
        return round(por_dia * len(e.lista_dias()) * 4.3 * ocupacion * self.REPARTO["anfitrión"])

    def ingresos_anfitrion(self, anfitrion_id):
        total = sum(r.cobro for r in self.reservas.values()
                    if r.anfitrion_id == anfitrion_id and r.estado in ("finalizada", "no_llego", "cancelada"))
        return round(total * self.REPARTO["anfitrión"])

    # ============================================================ CLIENTE
    def registrar_conductor(self, nombre, email, patente, tamano):
        if tamano not in ("pequeño", "mediano", "grande"):
            raise ErrorNegocio("El tamaño debe ser pequeño, mediano o grande.")
        for c in self.conductores.values():
            if email and c.email.lower() == email.lower():
                return c
        c = Conductor(self._nuevo_id("siguiente_conductor", "C"), nombre, email, patente.upper(), tamano)
        self.conductores[c.id] = c
        self._registrar("registro", detalle=f"Nuevo conductor {nombre}")
        self.guardar()
        return c

    def reservar(self, conductor_id, inicio, minutos, espacio_id="", invitados=""):
        """Obligatoria cliente (1/2). Verifica compatibilidad y que no haya choques (RN02, RN03).
        Sin espacio_id, la reserva queda 'por_asignar' para la administración."""
        c = self.conductores[conductor_id]
        if c.en_revision:
            raise ErrorNegocio("Tu cuenta está en revisión por no llegadas. La administración debe habilitarla.")
        termino = self._validar_periodo(inicio, minutos)
        anfitrion_id = ""
        if espacio_id:
            e = self.espacios.get(espacio_id)
            if not e:
                raise ErrorNegocio(f"No existe el estacionamiento {espacio_id}.")
            motivo = self.motivo_no_disponible(e, inicio, termino, c.tamano_auto)
            if motivo:
                raise ErrorNegocio(f"No se puede reservar {espacio_id}: {motivo}.")
            anfitrion_id = e.anfitrion_id
        r = Reserva(
            id=self._nuevo_id("siguiente_reserva", "R"), conductor_id=c.id, conductor_nombre=c.nombre,
            inicio=a_texto(inicio), termino=a_texto(termino), estacionamiento_id=espacio_id,
            anfitrion_id=anfitrion_id, estado="confirmada" if espacio_id else "por_asignar",
            monto=self.precio(inicio, termino), codigo_acceso=str(random.randint(1000, 9999)),
            creada=self.config["reloj"], invitados=invitados)
        self.reservas[r.id] = r
        self._registrar("creación", r, detalle="Reserva pagada" if espacio_id else "Pide asignación a la administración",
                        monto=r.monto)
        self._cal_crear(r)
        self.guardar()
        return r

    def _reserva_de(self, reserva_id, conductor_id):
        r = self.reservas.get(reserva_id)
        if not r or r.conductor_id != conductor_id:
            raise ErrorNegocio(f"No tienes una reserva {reserva_id}.")
        return r

    def cancelar(self, reserva_id, conductor_id):
        """Obligatoria cliente (2/2): cancelar. Antes del inicio se devuelve el 100 %; después, nada."""
        r = self._reserva_de(reserva_id, conductor_id)
        if r.estado not in ("confirmada", "por_asignar"):
            raise ErrorNegocio(f"La reserva está '{r.estado}': ya no se puede cancelar.")
        if self.ahora() < a_fecha(r.inicio):
            r.devolucion, r.cobro, detalle = r.monto, 0, "Cancelada antes del inicio: devolución del 100 %"
        else:
            r.devolucion, r.cobro, detalle = 0, r.monto, "Cancelada después del inicio: sin devolución"
        r.estado, r.motivo = "cancelada", detalle
        self._registrar("cancelación", r, detalle=detalle, monto=r.cobro)
        self._cal_cancelar(r)
        self.guardar()
        return r

    def registrar_llegada(self, reserva_id, conductor_id):
        r = self._reserva_de(reserva_id, conductor_id)
        if r.estado != "confirmada":
            raise ErrorNegocio(f"La reserva está '{r.estado}'.")
        ini, ahora = a_fecha(r.inicio), self.ahora()
        tol = timedelta(minutes=self.TOLERANCIA_MIN)
        if ahora < ini - tol:
            raise ErrorNegocio(f"Puedes registrar tu llegada desde las {a_texto(ini - tol)[11:]}.")
        if ahora > ini + tol:
            raise ErrorNegocio("Pasaron los 15 minutos de tolerancia.")
        r.estado, r.llegada = "en_uso", self.config["reloj"]
        self._registrar("llegada", r, detalle=f"Ingresó con el código {r.codigo_acceso}")
        self.guardar()
        return r

    def liberar_reserva(self, reserva_id, conductor_id):
        """Obligatoria cliente: liberar anticipadamente (paga lo reservado) o terminar
        con recargo si se pasó de la hora."""
        r = self._reserva_de(reserva_id, conductor_id)
        if r.estado != "en_uso":
            raise ErrorNegocio("Solo se puede liberar una reserva en uso. Si aún no llegas, usa cancelar.")
        ahora, fin = self.ahora(), a_fecha(r.termino)
        r.estado, r.fin_uso, r.cobro = "finalizada", self.config["reloj"], r.monto
        if ahora < fin:
            self._registrar("liberación", r, detalle="Liberó antes del término: paga lo reservado", monto=r.cobro)
        else:
            extra = int((ahora - fin).total_seconds() // 60)
            r.recargo = extra * self.RECARGO_MINUTO
            r.cobro += r.recargo
            self._registrar("término", r, detalle=f"Se pasó {extra} min: recargo ${r.recargo}", monto=r.cobro)
        self.guardar()
        return r

    def calificar(self, reserva_id, conductor_id, estrellas, comentario=""):
        """Propia cliente: calificar el espacio (confianza entre desconocidos).
        Regla: con 2 o más notas y promedio menor a 3, el espacio queda en revisión."""
        r = self._reserva_de(reserva_id, conductor_id)
        if r.estado != "finalizada":
            raise ErrorNegocio("Solo se califican reservas finalizadas.")
        if r.calificacion:
            raise ErrorNegocio("Esta reserva ya fue calificada.")
        if estrellas not in (1, 2, 3, 4, 5):
            raise ErrorNegocio("La calificación va de 1 a 5.")
        r.calificacion = estrellas
        self.calificaciones.append(Calificacion(self.config["reloj"], r.id, r.estacionamiento_id,
                                                conductor_id, estrellas, comentario))
        self._registrar("calificación", r, detalle=f"{estrellas} estrellas")
        notas = [x.estrellas for x in self.calificaciones if x.estacionamiento_id == r.estacionamiento_id]
        e = self.espacios[r.estacionamiento_id]
        promedio = sum(notas) / len(notas)
        if len(notas) >= 2 and promedio < 3 and not e.en_revision:
            e.en_revision = True
            self._registrar("revisión", espacio_id=e.id, detalle=f"Promedio {promedio:.1f}: deja de ofrecerse")
        self.guardar()
        return promedio

    def promedio(self, espacio_id):
        notas = [x.estrellas for x in self.calificaciones if x.estacionamiento_id == espacio_id]
        return sum(notas) / len(notas) if notas else None

    # ============================================================ ADMINISTRACIÓN
    def asignar(self, reserva_id, espacio_id):
        """Obligatoria administración (1/2): asociar una reserva a un espacio disponible."""
        r = self.reservas.get(reserva_id)
        if not r or r.estado != "por_asignar":
            raise ErrorNegocio(f"{reserva_id} no está esperando asignación.")
        self._mover(r, espacio_id)
        r.estado = "confirmada"
        self._registrar("asignación", r, detalle=f"Asignada a {espacio_id}")
        self._cal_crear(r)
        self.guardar()
        return r

    def reasignar(self, reserva_id, espacio_id, motivo="Contingencia"):
        """Obligatoria administración (2/2): mover una reserva vigente a otro espacio compatible."""
        r = self.reservas.get(reserva_id)
        if not r or not r.activa():
            raise ErrorNegocio(f"{reserva_id} no es una reserva vigente.")
        if r.estacionamiento_id == espacio_id:
            raise ErrorNegocio("La reserva ya está en ese espacio.")
        anterior = r.estacionamiento_id
        self._mover(r, espacio_id)
        r.motivo = f"{motivo}: de {anterior} a {espacio_id}"
        self._registrar("reasignación", r, detalle=r.motivo)
        self._cal_actualizar(r)
        self.guardar()
        return r

    def _mover(self, r, espacio_id):
        e = self.espacios.get(espacio_id)
        if not e:
            raise ErrorNegocio(f"No existe el estacionamiento {espacio_id}.")
        tamano = self.conductores[r.conductor_id].tamano_auto
        motivo = self.motivo_no_disponible(e, a_fecha(r.inicio), a_fecha(r.termino), tamano, r.id)
        if motivo:
            raise ErrorNegocio(f"No se puede usar {espacio_id}: {motivo}.")
        r.estacionamiento_id, r.anfitrion_id = e.id, e.anfitrion_id

    def procesar_vencidas(self):
        """Propia administración: penalizaciones.
        - No llegó en 15 min: se cobra el 30 % y se devuelve el 70 %.
        - 3 no llegadas en 30 días: el conductor queda en revisión.
        - Sin espacio asignado al llegar la hora: se devuelve el 100 %."""
        ahora, resumen = self.ahora(), []
        for r in self.reservas.values():
            if r.estado == "confirmada" and ahora > a_fecha(r.inicio) + timedelta(minutes=self.TOLERANCIA_MIN):
                r.estado = "no_llego"
                r.cobro = round(r.monto * self.COBRO_NO_LLEGADA)
                r.devolucion = r.monto - r.cobro
                self._registrar("no llegada", r, detalle=f"Cobro 30 % ${r.cobro}, devolución ${r.devolucion}", monto=r.cobro)
                c = self.conductores[r.conductor_id]
                c.agregar_no_llegada(r.inicio)
                limite = ahora - timedelta(days=30)
                recientes = [f for f in c.lista_no_llegadas() if a_fecha(f) >= limite]
                resumen.append(f"{r.id} de {c.nombre}: no llegó, se cobran ${r.cobro}")
                if len(recientes) >= self.NO_LLEGADAS_REVISION and not c.en_revision:
                    c.en_revision = True
                    self._registrar("revisión", r, detalle=f"{c.nombre}: {len(recientes)} no llegadas en 30 días")
                    resumen.append(f"{c.nombre} queda en revisión ({len(recientes)} no llegadas en 30 días)")
            elif r.estado == "por_asignar" and ahora >= a_fecha(r.inicio):
                r.estado, r.devolucion = "cancelada", r.monto
                r.motivo = "Sin espacio asignado a tiempo"
                self._registrar("cancelación", r, detalle=r.motivo + ": devolución del 100 %")
                resumen.append(f"{r.id}: cancelada sin espacio, devolución del 100 %")
        if resumen:
            self.guardar()
        return resumen

    def habilitar_conductor(self, conductor_id):
        c = self.conductores[conductor_id]
        if not c.en_revision:
            raise ErrorNegocio(f"{c.nombre} no está en revisión.")
        c.en_revision, c.fechas_no_llegada = False, ""
        self._registrar("habilitación", detalle=f"{c.nombre} puede volver a reservar")
        self.guardar()

    def aprobar_espacio(self, espacio_id):
        e = self.espacios[espacio_id]
        e.aprobado, e.en_revision = True, False
        self._registrar("aprobación", espacio_id=e.id, detalle="Espacio aprobado por la administración")
        self.guardar()

    # ============================================================ DEMO E INDICADORES
    def avanzar_reloj(self, minutos):
        self.config["reloj"] = a_texto(self.ahora() + timedelta(minutes=minutos))
        resumen = self.procesar_vencidas()
        self.guardar()
        return resumen

    def indicadores(self):
        rs = list(self.reservas.values())
        cerradas = [r for r in rs if r.estado in ("finalizada", "no_llego")]
        cobrado = sum(r.cobro for r in rs if r.estado in ("finalizada", "no_llego", "cancelada"))
        horas = sum(r.minutos() for r in rs if r.estado in ("finalizada", "en_uso", "confirmada")) / 60
        return {
            "Reservas totales": len(rs),
            "Horas reservadas": round(horas, 1),
            "Tasa de no llegada": f"{(sum(r.estado == 'no_llego' for r in cerradas) / len(cerradas) * 100) if cerradas else 0:.0f} %",
            "Reasignaciones": sum(1 for h in self.historial if h.tipo == "reasignación"),
            "Ingreso cobrado": cobrado,
            "Para anfitriones (60 %)": round(cobrado * self.REPARTO["anfitrión"]),
            "Para el edificio (25 %)": round(cobrado * self.REPARTO["edificio"]),
            "Para la plataforma (15 %)": round(cobrado * self.REPARTO["plataforma"]),
        }

In [ ]:
%%writefile /content/drive/MyDrive/Cupo/datos.py
"""
datos.py — Persistencia: todo el sistema se guarda en un Excel con una hoja por entidad.

Cada operación del sistema reescribe el archivo, así que al cerrar Colab y volver
a ejecutar, los datos siguen ahí (RT03). Las mismas hojas se pueden exportar a CSV
para el dashboard de la Entrega 3 (RT04).
"""
import os
from dataclasses import asdict, fields

import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill

from modelos import (Administrador, Anfitrion, Calificacion, Conductor, Estacionamiento,
                     Evento, Reserva)

HOJAS = {
    "Estacionamientos": Estacionamiento,
    "Anfitriones": Anfitrion,
    "Conductores": Conductor,
    "Administradores": Administrador,
    "Reservas": Reserva,
    "Historial": Evento,
    "Calificaciones": Calificacion,
}


def _convertir(valor, tipo):
    """Pasa un valor leído del Excel al tipo del campo de la clase."""
    if valor is None or (isinstance(valor, float) and pd.isna(valor)) or valor == "":
        return False if tipo is bool else 0 if tipo is int else ""
    if tipo is bool:
        return str(valor).strip().lower() in ("true", "1", "sí", "si", "verdadero")
    if tipo is int:
        return int(float(valor))
    texto = str(valor)
    return texto[:-2] if texto.endswith(".0") and texto[:-2].isdigit() else texto


class RepositorioExcel:
    def __init__(self, ruta):
        self.ruta = ruta

    def existe(self):
        return os.path.exists(self.ruta)

    def cargar(self):
        """Lee todas las hojas y devuelve listas de objetos."""
        libro = pd.read_excel(self.ruta, sheet_name=None, dtype=object)
        datos = {}
        for hoja, clase in HOJAS.items():
            tipos = {f.name: f.type for f in fields(clase)}
            df = libro.get(hoja, pd.DataFrame(columns=list(tipos)))
            datos[hoja] = [clase(**{k: _convertir(fila.get(k), t) for k, t in tipos.items()})
                           for fila in df.to_dict("records")]
        cfg = libro.get("Config", pd.DataFrame(columns=["clave", "valor"]))
        datos["Config"] = {str(f["clave"]): str(f["valor"]) for f in cfg.to_dict("records")}
        return datos

    def guardar(self, sistema):
        """Escribe todas las hojas con formato legible."""
        tablas = {
            "Estacionamientos": sistema.espacios.values(),
            "Anfitriones": sistema.anfitriones.values(),
            "Conductores": sistema.conductores.values(),
            "Administradores": sistema.administradores.values(),
            "Reservas": sistema.reservas.values(),
            "Historial": sistema.historial,
            "Calificaciones": sistema.calificaciones,
        }
        self.escribir(tablas, sistema.config)

    def escribir(self, tablas, config):
        temporal = self.ruta + ".tmp.xlsx"
        with pd.ExcelWriter(temporal, engine="openpyxl") as w:
            for hoja, clase in HOJAS.items():
                columnas = [f.name for f in fields(clase)]
                df = pd.DataFrame([asdict(o) for o in tablas[hoja]], columns=columnas)
                df.to_excel(w, sheet_name=hoja, index=False)
                self._formato(w.sheets[hoja], df)
            cfg = pd.DataFrame(sorted(config.items()), columns=["clave", "valor"])
            cfg.to_excel(w, sheet_name="Config", index=False)
            self._formato(w.sheets["Config"], cfg)
        try:
            os.replace(temporal, self.ruta)
        except OSError:            # algunas carpetas de Drive no permiten reemplazar
            os.remove(self.ruta) if os.path.exists(self.ruta) else None
            os.rename(temporal, self.ruta)

    @staticmethod
    def _formato(ws, df):
        encabezado = PatternFill("solid", start_color="262B30")
        for celda in ws[1]:
            celda.font = Font(name="Arial", bold=True, color="FFFFFF")
            celda.fill = encabezado
            celda.alignment = Alignment(vertical="center")
        for fila in ws.iter_rows(min_row=2):
            for celda in fila:
                celda.font = Font(name="Arial")
        for i, col in enumerate(df.columns, start=1):
            largo = max([len(str(col))] + [len(str(v)) for v in df[col].tolist()[:200]])
            ws.column_dimensions[ws.cell(1, i).column_letter].width = min(max(10, largo + 2), 60)
        ws.freeze_panes = "A2"

    def exportar_csv(self, sistema, carpeta):
        """Una hoja por archivo CSV, para el dashboard de la Entrega 3."""
        os.makedirs(carpeta, exist_ok=True)
        libro = pd.read_excel(self.ruta, sheet_name=None)
        rutas = []
        for hoja, df in libro.items():
            ruta = os.path.join(carpeta, f"{hoja.lower()}.csv")
            df.to_csv(ruta, index=False, encoding="utf-8-sig")
            rutas.append(ruta)
        return rutas

In [ ]:
%%writefile /content/drive/MyDrive/Cupo/calendario.py
"""
calendario.py — Integración con Google Calendar.

Con la cuenta de Google conectada, cada reserva crea un evento real con el conductor,
el anfitrión y los invitados como asistentes: Google les envía la invitación por correo
y el evento aparece en sus calendarios. Si se reubica o cancela, el evento se actualiza
o se elimina y Google avisa a todos.

Sin cuenta conectada, el sistema entrega un enlace que abre Google Calendar con el
evento ya lleno.
"""
import os
from urllib.parse import parse_qs, quote, urlparse

from modelos import a_fecha

os.environ.setdefault("OAUTHLIB_RELAX_TOKEN_SCOPE", "1")   # Google puede reordenar los permisos

PERMISOS = [
    "https://www.googleapis.com/auth/calendar.events",
    "openid",
    "https://www.googleapis.com/auth/userinfo.email",
    "https://www.googleapis.com/auth/userinfo.profile",
]
REDIRECCION = "http://localhost:8085/"


class CalendarioGoogle:
    def __init__(self, carpeta, zona_horaria="America/Santiago"):
        self.ruta_credenciales = os.path.join(carpeta, "client_secret.json")
        self.ruta_token = os.path.join(carpeta, "token_google.json")
        self.zona = zona_horaria
        self.servicio = None
        self.cuenta = ""
        self.nombre = ""

    @property
    def conectado(self):
        return self.servicio is not None

    # ------------------------------------------------------------ conexión
    def reconectar(self):
        """Usa el permiso guardado en Drive, si existe. No pide nada al usuario."""
        if not os.path.exists(self.ruta_token):
            return False
        try:
            from google.auth.transport.requests import Request
            from google.oauth2.credentials import Credentials
            cred = Credentials.from_authorized_user_file(self.ruta_token, PERMISOS)
            if not cred.valid and cred.refresh_token:
                cred.refresh(Request())
                self._guardar_token(cred)
            self._iniciar(cred)
            return True
        except Exception as err:
            print(f"  (No se pudo reutilizar el permiso guardado: {err})")
            return False

    def conectar(self):
        """Pide permiso a la cuenta de Google una sola vez y lo guarda en Drive."""
        if not os.path.exists(self.ruta_credenciales):
            raise FileNotFoundError(
                "Falta client_secret.json en la carpeta del proyecto. "
                "Revisa el paso 'Configurar Google Calendar' del cuaderno.")
        from google_auth_oauthlib.flow import InstalledAppFlow
        flujo = InstalledAppFlow.from_client_secrets_file(self.ruta_credenciales, PERMISOS,
                                                          redirect_uri=REDIRECCION)
        url, _ = flujo.authorization_url(prompt="consent", access_type="offline")
        print("\n1. Abre este enlace, elige tu cuenta de Google y acepta los permisos:\n")
        print(url)
        print("\n2. Al final verás una página que no carga (dice localhost). Es normal.")
        print("3. Copia la dirección COMPLETA de esa página y pégala aquí.\n")
        pegado = input("Dirección de la página: ").strip()
        codigo = parse_qs(urlparse(pegado).query).get("code", [None])[0]
        if not codigo:
            raise ValueError("Esa dirección no trae el código de autorización. Vuelve a intentarlo.")
        flujo.fetch_token(code=codigo)
        self._guardar_token(flujo.credentials)
        self._iniciar(flujo.credentials)

    def _guardar_token(self, cred):
        with open(self.ruta_token, "w") as f:
            f.write(cred.to_json())

    def _iniciar(self, cred):
        from googleapiclient.discovery import build
        self.servicio = build("calendar", "v3", credentials=cred, cache_discovery=False)
        info = build("oauth2", "v2", credentials=cred, cache_discovery=False).userinfo().get().execute()
        self.cuenta, self.nombre = info.get("email", ""), info.get("name", "")

    # ------------------------------------------------------------ eventos
    @staticmethod
    def invitados(reserva, anfitrion, conductor):
        correos = [conductor.email, anfitrion.email] + [x.strip() for x in reserva.invitados.split(",")]
        vistos = []
        for c in correos:
            if c and c.lower() not in [v.lower() for v in vistos]:
                vistos.append(c)
        return vistos

    @staticmethod
    def _texto(reserva, espacio, anfitrion, conductor):
        pagado = "$" + f"{reserva.monto:,}".replace(",", ".")
        return (f"Reserva {reserva.id} en Cupo\n"
                f"Espacio {espacio.id}, nivel {espacio.nivel}, {espacio.edificio}\n"
                f"Anfitrión: {anfitrion.nombre}\n"
                f"Conductor: {conductor.nombre} · patente {conductor.patente}\n"
                f"Código de acceso: {reserva.codigo_acceso}\n"
                f"Pagado: {pagado}\n\n"
                "Reglas: si cancelas antes del inicio se devuelve el 100 %. Hay 15 min de tolerancia; "
                "si no llegas se cobra el 30 % y se devuelve el 70 %.\n"
                f"Reglas del espacio: {espacio.reglas or 'sin reglas especiales'}")

    def _cuerpo(self, reserva, espacio, anfitrion, conductor):
        return {
            "summary": f"Estacionamiento {espacio.id} · Cupo",
            "location": f"{espacio.edificio}, {espacio.direccion}",
            "description": self._texto(reserva, espacio, anfitrion, conductor),
            "start": {"dateTime": a_fecha(reserva.inicio).strftime("%Y-%m-%dT%H:%M:00"), "timeZone": self.zona},
            "end": {"dateTime": a_fecha(reserva.termino).strftime("%Y-%m-%dT%H:%M:00"), "timeZone": self.zona},
            "attendees": [{"email": c} for c in self.invitados(reserva, anfitrion, conductor)],
            "reminders": {"useDefault": False, "overrides": [{"method": "popup", "minutes": 30},
                                                             {"method": "email", "minutes": 30}]},
        }

    def crear_evento(self, reserva, espacio, anfitrion, conductor):
        evento = self.servicio.events().insert(
            calendarId="primary", body=self._cuerpo(reserva, espacio, anfitrion, conductor),
            sendUpdates="all").execute()
        return evento["id"]

    def actualizar_evento(self, reserva, espacio, anfitrion, conductor):
        self.servicio.events().patch(
            calendarId="primary", eventId=reserva.evento_google,
            body=self._cuerpo(reserva, espacio, anfitrion, conductor), sendUpdates="all").execute()

    def cancelar_evento(self, evento_id):
        self.servicio.events().delete(calendarId="primary", eventId=evento_id, sendUpdates="all").execute()

    def enlace_plantilla(self, reserva, espacio, anfitrion, conductor):
        """Enlace de respaldo: abre Google Calendar con el evento lleno y los invitados."""
        fechas = (a_fecha(reserva.inicio).strftime("%Y%m%dT%H%M00") + "/"
                  + a_fecha(reserva.termino).strftime("%Y%m%dT%H%M00"))
        partes = [
            "action=TEMPLATE",
            "text=" + quote(f"Estacionamiento {espacio.id} · Cupo"),
            "dates=" + fechas,
            "ctz=" + quote(self.zona),
            "details=" + quote(self._texto(reserva, espacio, anfitrion, conductor)),
            "location=" + quote(f"{espacio.edificio}, {espacio.direccion}"),
        ]
        correos = self.invitados(reserva, anfitrion, conductor)
        if correos:
            partes.append("add=" + quote(",".join(correos)))
        return "https://calendar.google.com/calendar/render?" + "&".join(partes)

In [ ]:
%%writefile /content/drive/MyDrive/Cupo/datos_demo.py
"""
datos_demo.py — Crea el Excel inicial con datos de prueba para la demostración.

Fecha de la demo: hoy (o el próximo día hábil), con la hora del sistema a las 14:00.
"""
from datetime import datetime, timedelta

from modelos import (Administrador, Anfitrion, Calificacion, Conductor, Estacionamiento,
                     Evento, Reserva, a_texto)

DIRECCION = "Av. Los Aromos 1450 (dirección de prueba)"


def _dia_habil(d):
    while d.weekday() >= 5:
        d += timedelta(days=1)
    return d


def crear_demo(repositorio):
    hoy = _dia_habil(datetime.now().replace(hour=0, minute=0, second=0, microsecond=0))
    ayer = hoy - timedelta(days=1)
    while ayer.weekday() >= 5:
        ayer -= timedelta(days=1)
    D, A = hoy.strftime("%Y-%m-%d"), ayer.strftime("%Y-%m-%d")

    anfitriones = [
        Anfitrion("A001", "Carla Muñoz", "", "Torre A, depto 504"),
        Anfitrion("A002", "Javiera Soto", "", "Torre B, depto 210"),
        Anfitrion("A003", "Rodrigo Fuentes", "", "Torre A, depto 802"),
        Anfitrion("A004", "Ignacio Pérez", "", "Torre B, depto 115"),
        Anfitrion("A005", "Paula Contreras", "", "Torre B, depto 703"),
        Anfitrion("A006", "Sofía Ramírez", "", "Torre A, depto 301"),
    ]
    E = lambda i, a, torre, nivel, tam, tech, carg, d, h, reglas, **kw: Estacionamiento(
        i, a, f"Edificio Los Aromos, {torre}", DIRECCION, nivel, tam, tech, carg,
        hora_desde=d, hora_hasta=h, reglas=reglas, **kw)
    espacios = [
        E("E05", "A001", "Torre A", "-1", "mediano", True, False, "09:00", "18:00", "No dejar objetos en el espacio."),
        E("E12", "A002", "Torre B", "-2", "mediano", True, False, "08:30", "17:30", "Entrar por el portón lateral."),
        E("E18", "A003", "Torre A", "-1", "grande", True, True, "09:00", "18:00", "Usar el cargador solo con permiso."),
        E("E21", "A004", "Torre B", "-2", "pequeño", True, False, "09:00", "18:00", "Espacio angosto."),
        E("E27", "A005", "Torre B", "-1", "grande", False, False, "10:00", "18:00", "Descubierto: no lavar el auto."),
        E("E40", "A006", "Torre A", "-1", "grande", True, False, "09:00", "18:00", "", aprobado=False),
    ]
    conductores = [
        Conductor("C001", "Camila Torres", "", "KXLT21", "mediano"),
        Conductor("C002", "Valentina Rojas", "", "PRZD48", "grande"),
        Conductor("C003", "Tomás Vera", "", "JHGS90", "mediano",
                  fechas_no_llegada=f"{a_texto(hoy - timedelta(days=6))[:10]} 10:00;{a_texto(hoy - timedelta(days=13))[:10]} 10:00"),
        Conductor("C004", "Martín Díaz", "", "FBWC12", "grande"),
    ]
    administradores = [Administrador("ADM1", "Administración Edificio Los Aromos", "")]
    reservas = [
        Reserva("R0001", "C001", "Camila Torres", f"{A} 10:00", f"{A} 12:00", "E05", "A001", "finalizada",
                2800, 2800, codigo_acceso="4821", creada=f"{A} 08:40", llegada=f"{A} 09:55",
                fin_uso=f"{A} 11:58", calificacion=5),
        Reserva("R0002", "C002", "Valentina Rojas", f"{D} 15:00", f"{D} 17:00", "E18", "A003", "confirmada",
                3600, codigo_acceso="7310", creada=f"{D} 09:12"),
        Reserva("R0003", "C003", "Tomás Vera", f"{D} 14:00", f"{D} 15:00", "E12", "A002", "confirmada",
                1400, codigo_acceso="5527", creada=f"{D} 10:30"),
        Reserva("R0004", "C004", "Martín Díaz", f"{D} 16:00", f"{D} 17:00", "", "", "por_asignar",
                1800, codigo_acceso="9046", creada=f"{D} 11:05"),
    ]
    historial = [
        Evento(f"{A} 08:40", "creación", "R0001", "E05", "A001", "C001", "confirmada", f"{A} 10:00", f"{A} 12:00", 2800, "Reserva pagada"),
        Evento(f"{A} 09:55", "llegada", "R0001", "E05", "A001", "C001", "en_uso", f"{A} 10:00", f"{A} 12:00", 0, "Ingresó con el código 4821"),
        Evento(f"{A} 11:58", "liberación", "R0001", "E05", "A001", "C001", "finalizada", f"{A} 10:00", f"{A} 12:00", 2800, "Liberó antes del término"),
        Evento(f"{A} 12:05", "calificación", "R0001", "E05", "A001", "C001", "finalizada", f"{A} 10:00", f"{A} 12:00", 0, "5 estrellas"),
        Evento(f"{D} 09:12", "creación", "R0002", "E18", "A003", "C002", "confirmada", f"{D} 15:00", f"{D} 17:00", 3600, "Reserva pagada"),
        Evento(f"{D} 10:30", "creación", "R0003", "E12", "A002", "C003", "confirmada", f"{D} 14:00", f"{D} 15:00", 1400, "Reserva pagada"),
        Evento(f"{D} 11:05", "creación", "R0004", "", "", "C004", "por_asignar", f"{D} 16:00", f"{D} 17:00", 1800, "Pide asignación a la administración"),
    ]
    calificaciones = [Calificacion(f"{A} 12:05", "R0001", "E05", "C001", 5, "Muy fácil de encontrar")]
    config = {"reloj": f"{D} 14:00", "siguiente_reserva": "5", "siguiente_conductor": "5",
              "zona_horaria": "America/Santiago"}
    repositorio.escribir({
        "Estacionamientos": espacios, "Anfitriones": anfitriones, "Conductores": conductores,
        "Administradores": administradores, "Reservas": reservas, "Historial": historial,
        "Calificaciones": calificaciones}, config)

In [ ]:
%%writefile /content/drive/MyDrive/Cupo/main.py
"""
main.py — Menú en consola de Cupo. Funciona en Google Colab y en cualquier Python 3.

Uso en Colab:  import main; main.ejecutar("/content/drive/MyDrive/Cupo")
"""
import os
from datetime import datetime, timedelta

from calendario import CalendarioGoogle
from datos import RepositorioExcel
from datos_demo import crear_demo
from modelos import DIAS_SEMANA, a_fecha
from reglas import ErrorNegocio, Sistema

LINEA = "─" * 60
ESTADOS = {"confirmada": "Confirmada", "en_uso": "En uso", "finalizada": "Finalizada",
           "cancelada": "Cancelada", "no_llego": "No llegó", "por_asignar": "Esperando asignación"}


def pesos(n):
    return "$" + f"{int(n):,}".replace(",", ".")


def pedir(texto, opciones=None, defecto=None):
    while True:
        extra = f" [{defecto}]" if defecto not in (None, "") else ""
        valor = input(f"{texto}{extra}: ").strip()
        if not valor and defecto is not None:
            valor = str(defecto)
        if opciones is None or valor in opciones:
            return valor
        print(f"  Opción no válida. Elige entre: {', '.join(opciones)}")


def pedir_hora(texto, defecto):
    while True:
        v = pedir(texto, defecto=defecto)
        try:
            datetime.strptime(v, "%H:%M")
            return v
        except ValueError:
            print("  Escribe la hora como HH:MM, por ejemplo 15:00.")


def pedir_fecha(texto, defecto):
    while True:
        v = pedir(texto, defecto=defecto)
        try:
            datetime.strptime(v, "%Y-%m-%d")
            return v
        except ValueError:
            print("  Escribe la fecha como AAAA-MM-DD.")


def elegir(titulo, items, etiqueta):
    """Muestra una lista numerada y devuelve el elemento elegido (o None)."""
    if not items:
        print("  No hay opciones disponibles.")
        return None
    print(f"\n{titulo}")
    for i, x in enumerate(items, 1):
        print(f"  {i}. {etiqueta(x)}")
    v = pedir("Número (Enter para volver)", [str(i) for i in range(1, len(items) + 1)] + [""], "")
    return items[int(v) - 1] if v else None


def mostrar_avisos(s):
    for a in s.avisos:
        print("  ▸ " + a)
    s.avisos.clear()


def operar(s, funcion, *args, ok=None):
    """Ejecuta una operación del sistema y muestra el resultado o el error de negocio."""
    try:
        resultado = funcion(*args)
        if ok:
            print("  ✔ " + (ok(resultado) if callable(ok) else ok))
        mostrar_avisos(s)
        return resultado
    except ErrorNegocio as e:
        print("  ✘ Operación rechazada: " + str(e))
        mostrar_avisos(s)
        return None


def linea_reserva(s, r):
    lugar = r.estacionamiento_id or "sin espacio"
    return f"{r.id} · {lugar} · {r.inicio} a {r.termino[11:]} · {pesos(r.monto)} · {ESTADOS[r.estado]}"


# ================================================================== CONDUCTOR
def identificar_conductor(s):
    cal = s.calendario
    if cal and cal.conectado:
        existente = next((c for c in s.conductores.values() if c.email.lower() == cal.cuenta.lower()), None)
        if existente:
            print(f"  Entraste con tu cuenta de Google: {existente.nombre} ({existente.email})")
            return existente
        if pedir(f"¿Registrarte con tu cuenta de Google {cal.cuenta}? (s/n)", ["s", "n"], "s") == "s":
            patente = pedir("Patente del auto")
            tam = pedir("Tamaño del auto (pequeño/mediano/grande)", ["pequeño", "mediano", "grande"], "mediano")
            return operar(s, s.registrar_conductor, cal.nombre or cal.cuenta, cal.cuenta, patente, tam,
                          ok="Registro hecho con los datos de tu cuenta de Google.")
    opciones = list(s.conductores.values()) + ["nuevo"]
    c = elegir("¿Quién eres?", opciones,
               lambda x: "Registrar un conductor nuevo" if x == "nuevo"
               else f"{x.nombre} · {x.patente} · auto {x.tamano_auto}{' · EN REVISIÓN' if x.en_revision else ''}")
    if c == "nuevo":
        nombre = pedir("Nombre")
        email = pedir("Correo (ahí llega la invitación de Google Calendar)")
        patente = pedir("Patente")
        tam = pedir("Tamaño del auto (pequeño/mediano/grande)", ["pequeño", "mediano", "grande"], "mediano")
        c = operar(s, s.registrar_conductor, nombre, email, patente, tam, ok="Conductor registrado.")
    return c


def menu_conductor(s):
    c = identificar_conductor(s)
    if not c:
        return
    while True:
        print(f"\n{LINEA}\nCONDUCTOR · {c.nombre} · auto {c.tamano_auto} · {s.config['reloj']}\n{LINEA}")
        print("1. Buscar y reservar un espacio\n2. Pedir que la administración me asigne uno"
              "\n3. Ver mis reservas\n4. Cancelar una reserva\n5. Registrar mi llegada"
              "\n6. Liberar el espacio / terminar el uso\n7. Calificar un espacio\n0. Volver")
        op = pedir("Opción", list("01234567"))
        mias = [r for r in s.reservas.values() if r.conductor_id == c.id]
        if op == "0":
            return
        if op in ("1", "2"):
            hoy = s.config["reloj"][:10]
            fecha = pedir_fecha("Día (AAAA-MM-DD)", hoy)
            hora = pedir_hora("Hora de inicio (HH:MM)", "15:00")
            minutos = int(pedir("Duración en minutos (30, 60, 90, 120)", ["30", "60", "90", "120"], "60"))
            inicio = a_fecha(f"{fecha} {hora}")
            invitados = pedir("Correos para invitar al evento, separados por coma (opcional)", defecto="")
            if op == "2":
                operar(s, s.reservar, c.id, inicio, minutos, "", invitados,
                       ok=lambda r: f"Solicitud {r.id} creada por {pesos(r.monto)}. La administración te asignará un espacio.")
                continue
            fin = inicio + timedelta(minutes=minutos)
            print(f"\nEspacios para {fecha} de {hora} a {fin.strftime('%H:%M')}:")
            for e in s.espacios.values():
                if not e.aprobado:
                    continue
                motivo = s.motivo_no_disponible(e, inicio, fin, c.tamano_auto)
                p = s.promedio(e.id)
                estrellas = f"★ {p:.1f}" if p else "sin calificar"
                print(f"  {'✔' if not motivo else '✘'} {e.describir()} · {estrellas}"
                      + (f"\n      no disponible: {motivo}" if motivo else f" · {pesos(s.precio(inicio, fin))}"))
            libres = s.espacios_compatibles(inicio, fin, c.tamano_auto)
            e = elegir("Elige un espacio disponible", libres, lambda x: f"{x.id} · {pesos(s.precio(inicio, fin))} · {x.reglas}")
            if not e:
                continue
            print("\nReglas: si cancelas antes del inicio se devuelve el 100 %. Hay 15 min de tolerancia;"
                  " si no llegas se cobra el 30 %. Si te pasas de la hora pagas $30 por minuto.")
            if pedir("¿Aceptas las reglas y confirmas el pago? (s/n)", ["s", "n"], "s") == "s":
                operar(s, s.reservar, c.id, inicio, minutos, e.id, invitados,
                       ok=lambda r: f"Reserva {r.id} confirmada en {r.estacionamiento_id}. Código de acceso: {r.codigo_acceso}.")
        elif op == "3":
            print()
            for r in sorted(mias, key=lambda x: x.inicio):
                print("  " + linea_reserva(s, r) + (f" · código {r.codigo_acceso}" if r.activa() else ""))
            if not mias:
                print("  Todavía no tienes reservas.")
        elif op == "4":
            r = elegir("¿Cuál cancelas?", [x for x in mias if x.estado in ("confirmada", "por_asignar")], lambda x: linea_reserva(s, x))
            if r:
                operar(s, s.cancelar, r.id, c.id, ok=lambda x: x.motivo)
        elif op == "5":
            r = elegir("¿A cuál llegaste?", [x for x in mias if x.estado == "confirmada"], lambda x: linea_reserva(s, x))
            if r:
                operar(s, s.registrar_llegada, r.id, c.id, ok=f"Llegada registrada en {r.estacionamiento_id}.")
        elif op == "6":
            r = elegir("¿Cuál liberas?", [x for x in mias if x.estado == "en_uso"], lambda x: linea_reserva(s, x))
            if r:
                operar(s, s.liberar_reserva, r.id, c.id,
                       ok=lambda x: f"Espacio liberado. Total cobrado {pesos(x.cobro)}" + (f" (recargo {pesos(x.recargo)})" if x.recargo else "."))
        elif op == "7":
            r = elegir("¿Qué reserva calificas?", [x for x in mias if x.estado == "finalizada" and not x.calificacion], lambda x: linea_reserva(s, x))
            if r:
                n = int(pedir("Estrellas (1 a 5)", list("12345")))
                comentario = pedir("Comentario (opcional)", defecto="")
                operar(s, s.calificar, r.id, c.id, n, comentario,
                       ok=lambda p: f"Gracias. Promedio actual de {r.estacionamiento_id}: {p:.1f}")


# ================================================================== ANFITRIÓN
def menu_anfitrion(s):
    a = elegir("¿Qué anfitrión eres?", list(s.anfitriones.values()), lambda x: f"{x.nombre} · {x.departamento}")
    if not a:
        return
    while True:
        mios = [e for e in s.espacios.values() if e.anfitrion_id == a.id]
        print(f"\n{LINEA}\nANFITRIÓN · {a.nombre} · {s.config['reloj']}\n{LINEA}")
        for e in mios:
            estado = "esperando aprobación" if not e.aprobado else e.estado
            print(f"  {e.describir()} · {estado.upper()}")
        print("\n1. Liberar mi espacio\n2. Bloquear mi espacio (lo voy a usar)"
              "\n3. Calendario inteligente: definir horario recurrente\n4. Ver reservas e ingresos"
              "\n5. Guardar mi correo para recibir invitaciones\n0. Volver")
        op = pedir("Opción", list("012345"))
        if op == "0":
            return
        if op in ("1", "2", "3"):
            e = mios[0] if len(mios) == 1 else elegir("¿Qué espacio?", mios, lambda x: x.id)
            if not e:
                continue
        if op == "1":
            operar(s, s.liberar_espacio, e.id, a.id, ok=f"{e.id} liberado: ya aparece para los conductores.")
        elif op == "2":
            motivo = pedir("Motivo", defecto="Uso del dueño")
            res = operar(s, s.bloquear_espacio, e.id, a.id, motivo, ok=f"{e.id} bloqueado.")
            for linea in res or []:
                print("    · " + linea)
            if res == []:
                print("    · No había reservas vigentes.")
        elif op == "3":
            print("\n  Según el modelo D(t), entre 09:00 y 18:00 casi todos los dueños están fuera."
                  "\n  Antes de las 08:30 aún no salen y después de las 18:00 vuelven.")
            if pedir("¿Usar el horario sugerido, lunes a viernes de 09:00 a 18:00? (s/n)", ["s", "n"], "s") == "s":
                desde, hasta, dias = "09:00", "18:00", [0, 1, 2, 3, 4]
            else:
                desde = pedir_hora("Desde (HH:MM)", e.hora_desde)
                hasta = pedir_hora("Hasta (HH:MM)", e.hora_hasta)
                print("  Días: " + ", ".join(f"{i}={d}" for i, d in enumerate(DIAS_SEMANA)))
                texto = pedir("Números de los días separados por coma", defecto=e.dias)
                dias = sorted({int(x) for x in texto.split(",") if x.strip().isdigit() and int(x) <= 6})
            avisos = operar(s, s.definir_horario, e.id, a.id, desde, hasta, dias, ok="Horario guardado.")
            for av in avisos or []:
                print("    ⚠ " + av)
            if avisos is not None:
                print(f"    Ganancia estimada: {pesos(s.ganancia_estimada(e.id))} al mes (60 % para ti, con 60 % de ocupación).")
        elif op == "4":
            rs = [r for r in s.reservas.values() if r.anfitrion_id == a.id]
            for r in sorted(rs, key=lambda x: x.inicio):
                print("  " + linea_reserva(s, r) + f" · {r.conductor_nombre}")
            print(f"  Ingresos para ti (60 %): {pesos(s.ingresos_anfitrion(a.id))}")
        elif op == "5":
            correo = pedir("Correo", defecto=a.email)
            a.email = correo
            s.guardar()
            print("  ✔ Correo guardado: te llegarán las invitaciones de tus reservas.")


# ================================================================== ADMINISTRACIÓN
def menu_admin(s):
    while True:
        print(f"\n{LINEA}\nADMINISTRACIÓN · {s.config['reloj']}\n{LINEA}")
        print("1. Ver reservas\n2. Asignar un espacio a una solicitud\n3. Reasignar una reserva vigente"
              "\n4. Penalizaciones: procesar no llegadas\n5. Habilitar un conductor en revisión"
              "\n6. Aprobar un espacio nuevo\n7. Ver registro histórico\n8. Indicadores\n0. Volver")
        op = pedir("Opción", list("012345678"))
        if op == "0":
            return
        if op == "1":
            for r in sorted(s.reservas.values(), key=lambda x: x.inicio):
                print("  " + linea_reserva(s, r) + f" · {r.conductor_nombre}")
        elif op in ("2", "3"):
            filtro = (lambda x: x.estado == "por_asignar") if op == "2" else (lambda x: x.activa())
            r = elegir("¿Qué reserva?", [x for x in s.reservas.values() if filtro(x)],
                       lambda x: linea_reserva(s, x) + f" · {x.conductor_nombre}")
            if not r:
                continue
            tam = s.conductores[r.conductor_id].tamano_auto
            opciones = s.espacios_compatibles(a_fecha(r.inicio), a_fecha(r.termino), tam, r.id, r.estacionamiento_id)
            print(f"  Compatibles (libres, en horario, sin choques, auto {tam}): {', '.join(e.id for e in opciones) or 'ninguno'}")
            destino = pedir("Espacio de destino (puedes escribir otro para probar la validación)").upper()
            if op == "2":
                operar(s, s.asignar, r.id, destino, ok=f"{r.id} asignada a {destino}.")
            else:
                motivo = pedir("Motivo", defecto="Contingencia")
                operar(s, s.reasignar, r.id, destino, motivo, ok=f"{r.id} reasignada a {destino}.")
        elif op == "4":
            res = s.procesar_vencidas()
            print("\n".join("  · " + x for x in res) if res else "  No hay reservas vencidas.")
            for c in s.conductores.values():
                if c.lista_no_llegadas():
                    print(f"  {c.nombre}: {len(c.lista_no_llegadas())} no llegada(s){' · EN REVISIÓN' if c.en_revision else ''}")
        elif op == "5":
            c = elegir("¿A quién habilitas?", [x for x in s.conductores.values() if x.en_revision], lambda x: x.nombre)
            if c:
                operar(s, s.habilitar_conductor, c.id, ok=f"{c.nombre} puede volver a reservar.")
        elif op == "6":
            e = elegir("¿Qué espacio apruebas?", [x for x in s.espacios.values() if not x.aprobado or x.en_revision], lambda x: x.describir())
            if e:
                operar(s, s.aprobar_espacio, e.id, ok=f"{e.id} aprobado.")
        elif op == "7":
            for h in s.historial[-25:]:
                print(f"  {h.fecha_hora} · {h.tipo:<22} · {h.reserva_id or '-':<5} · {h.estacionamiento_id or '-':<3} · {h.detalle}")
            print(f"  (Se muestran los últimos 25 de {len(s.historial)}. Todos están en la hoja Historial del Excel.)")
        elif op == "8":
            for k, v in s.indicadores().items():
                print(f"  {k:<28} {pesos(v) if isinstance(v, int) and 'Ingreso' in k or 'Para' in k else v}")


# ================================================================== PRINCIPAL
def ejecutar(carpeta="."):
    os.makedirs(carpeta, exist_ok=True)
    repo = RepositorioExcel(os.path.join(carpeta, "cupo_datos.xlsx"))
    if not repo.existe():
        crear_demo(repo)
        print("Se creó cupo_datos.xlsx con los datos de prueba.")
    calendario = CalendarioGoogle(carpeta)
    calendario.reconectar()
    s = Sistema(repo, calendario)
    while True:
        cuenta = f"conectado como {calendario.cuenta}" if calendario.conectado else "no conectado (se usan enlaces)"
        print(f"\n{'═' * 60}\nCUPO · el Airbnb de los estacionamientos\n"
              f"Hora del sistema: {s.config['reloj']} · Google Calendar: {cuenta}\n{'═' * 60}")
        print("1. Conductor\n2. Anfitrión\n3. Administración\n4. Avanzar la hora (demo)"
              "\n5. Conectar mi cuenta de Google Calendar\n6. Exportar los datos a CSV\n0. Salir")
        op = pedir("Opción", list("0123456"))
        if op == "0":
            print(f"Datos guardados en {repo.ruta}. ¡Hasta pronto!")
            return
        if op == "1":
            menu_conductor(s)
        elif op == "2":
            menu_anfitrion(s)
        elif op == "3":
            menu_admin(s)
        elif op == "4":
            m = int(pedir("¿Cuántos minutos? (15, 30, 60, 120)", ["15", "30", "60", "120"], "30"))
            res = s.avanzar_reloj(m)
            print(f"  Hora del sistema: {s.config['reloj']}")
            for x in res:
                print("  · " + x)
        elif op == "5":
            try:
                calendario.conectar()
                print(f"  ✔ Conectado como {calendario.cuenta}. Las reservas crearán eventos e invitaciones reales.")
            except Exception as e:
                print(f"  ✘ No se pudo conectar: {e}")
        elif op == "6":
            rutas = repo.exportar_csv(s, os.path.join(carpeta, "csv"))
            print("  ✔ Archivos creados:\n" + "\n".join("    " + r for r in rutas))


if __name__ == "__main__":
    ejecutar(os.path.dirname(os.path.abspath(__file__)))

## Paso 4 · Configurar Google Calendar (una sola vez, opcional)
Sin este paso el sistema funciona igual, pero en vez de crear el evento solo, muestra un enlace que abre Google Calendar con el evento lleno.

1. Entra a [console.cloud.google.com](https://console.cloud.google.com) con la cuenta que hará la demo y crea un proyecto (por ejemplo, *Cupo*).
2. En **APIs y servicios › Biblioteca**, busca **Google Calendar API** y actívala.
3. En **Pantalla de consentimiento de OAuth** (o **Google Auth Platform**), elige **Externo**, ponle nombre *Cupo* y tu correo. En **Usuarios de prueba**, agrega los correos que van a conectar su cuenta (los tres integrantes).
4. En **Credenciales › Crear credenciales › ID de cliente de OAuth**, elige **App de escritorio** y descarga el archivo JSON.
5. Ejecuta la celda de abajo y sube ese archivo. Queda guardado como `client_secret.json` en la carpeta del proyecto.

Después, en el menú del sistema, elige **5. Conectar mi cuenta de Google Calendar**. Google mostrará *"Google no verificó esta app"*: toca **Continuar**, porque la app es tuya y está en modo de prueba. El permiso queda guardado en Drive; en modo de prueba, Google lo pide de nuevo cada 7 días.

In [ ]:
from google.colab import files
subido = files.upload()
for nombre, contenido in subido.items():
    with open(os.path.join(CARPETA, 'client_secret.json'), 'wb') as f:
        f.write(contenido)
    print('Guardado como client_secret.json')

## Paso 5 · Ejecutar el sistema
Si no existe `cupo_datos.xlsx`, se crea con datos de prueba (hora del sistema: 14:00 del día hábil de hoy). Cada operación se guarda al instante en el Excel.

In [ ]:
for m in ['main', 'reglas', 'datos', 'calendario', 'datos_demo', 'modelos']:
    sys.modules.pop(m, None)
import main
main.ejecutar(CARPETA)

## Paso 6 · Descargar el Excel
También puedes abrirlo directamente desde Google Drive, en **Mi unidad › Cupo**.

In [ ]:
files.download(os.path.join(CARPETA, 'cupo_datos.xlsx'))

## Reiniciar la demo
Borra el Excel para volver a los datos de prueba la próxima vez que ejecutes el paso 5.

In [ ]:
ruta = os.path.join(CARPETA, 'cupo_datos.xlsx')
if os.path.exists(ruta):
    os.remove(ruta)
    print('Excel borrado: el paso 5 creará los datos de prueba de nuevo.')

## Guion de la demostración
1. **Cliente (obligatoria):** menú 1 › registrarse › 1. Buscar y reservar › hoy 15:00, 60 min › agregar el correo de la profesora › elegir **E05**. Con Google conectado, el evento y las invitaciones salen solos.
2. **Propietario (obligatoria + contingencia):** menú 2 › Carla Muñoz › 2. Bloquear. La reserva se reubica en otro espacio compatible y el evento de Google se actualiza.
3. **Propietario (propia):** 3. Calendario inteligente › horario sugerido por D(t) y ganancia estimada.
4. **Penalizaciones (propia de administración):** menú 4 › 30 min. Tomás Vera no llegó: se cobra el 30 % y, por su tercera no llegada, queda en revisión.
5. **Administración (obligatoria):** menú 3 › 3. Reasignar › escribir **E21** para mostrar el rechazo (operación inválida) › luego 2. Asignar a Martín Díaz en **E27**.
6. **Cliente (obligatoria + propia):** menú 4 › 15 min › registrar llegada › liberar el espacio › calificar.
7. **Datos:** menú 3 › 7. Registro histórico y 8. Indicadores; abrir el Excel en Drive.